# ДЗ 5 · CTE, рекурсія та Materialized View на Olist
PostgreSQL обчислює повні customer-level features; Python запускає сервер, читає CSV та показує результати.

## 1. Setup і PostgreSQL
Перша клітинка встановлює відсутні зафіксовані залежності, запускає власний тимчасовий PostgreSQL через `pgserver` та показує фактичну версію. Після Restart & Run All остання клітинка зупиняє й видаляє тільки цей кластер. Кожний повний запуск перебудовує власну схему `hw5`; в інших базах нічого не змінюється.

In [1]:
import sys, subprocess, importlib.metadata
required = {'pgserver':'0.1.4','psycopg2-binary':'2.9.12','SQLAlchemy':'2.0.54','pandas':'2.3.3'}
missing=[]
for package,version in required.items():
    try: installed=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError: installed=None
    if installed!=version: missing.append(f'{package}=={version}')
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
package_versions={package:importlib.metadata.version(package) for package in required}
print('Python:',sys.version.split()[0],'; packages:',package_versions)
"""Notebook-embedded local runtime. No connection to external database servers."""
from pathlib import Path
import atexit, csv, hashlib, html, io, json, re, tempfile, time, urllib.request, zipfile
from decimal import Decimal
import pandas as pd
import pgserver
from psycopg2 import sql as pgsql
from sqlalchemy import create_engine, text
from IPython.display import display, Markdown, HTML

pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth',None)
results={}; checks=[]; plans={}; profile_rows=[]
cwd=Path.cwd().resolve()
PROJECT_DIR=cwd.parent if cwd.name=='deliverables' else cwd
WORK_DIR=PROJECT_DIR/'work'; WORK_DIR.mkdir(parents=True,exist_ok=True)
PLAN_DIR=WORK_DIR/'plans'; PLAN_DIR.mkdir(exist_ok=True)
course_dir=next((p for p in (PROJECT_DIR,*PROJECT_DIR.parents) if p.name=='Relational Databases for Data Science'),None)
cluster_directory=tempfile.TemporaryDirectory(prefix='hw5-olist-')
pg=pgserver.get_server(Path(cluster_directory.name)/'pgdata',cleanup_mode='stop')
engine=create_engine(pg.get_uri(),future=True,connect_args={'options':'-c search_path=hw5,public -c statement_timeout=120000'})
_cleaned=False
def cleanup_hw5():
    global _cleaned
    if not _cleaned:
        engine.dispose(); pg.cleanup(); cluster_directory.cleanup(); _cleaned=True
atexit.register(cleanup_hw5)
def json_default(value):
    if hasattr(value,'isoformat'): return value.isoformat()
    if isinstance(value,Decimal): return str(value)
    return str(value)
def write_json(path,value):
    path.write_text(json.dumps(value,ensure_ascii=False,indent=2,default=json_default)+'\n',encoding='utf-8')
def rows(statement,connection=None):
    if connection is not None: return [dict(r) for r in connection.execute(text(statement)).mappings()]
    with engine.connect() as connection: return rows(statement,connection)
def scalar(statement,connection=None):
    if connection is not None: return connection.execute(text(statement)).scalar_one()
    with engine.connect() as connection: return scalar(statement,connection)
def execute_sql(statement):
    with engine.begin() as connection: connection.execute(text(statement))
def sql_display(statement):
    display(Markdown('```sql\n'+statement.strip()+'\n```'))
def table_display(data,limit=10):
    if not data: print('0 рядків'); return
    frame=pd.DataFrame(data)
    if len(frame)==1 and len(frame.columns)>7:
        markup=frame.T.rename(columns={0:'value'}).rename_axis('metric').to_html(na_rep='NULL',escape=True)
        display(HTML('<div style="overflow-x:auto;max-width:100%">'+markup+'</div>'))
    else:
        markup=frame.head(limit).to_html(index=False,na_rep='NULL',escape=True)
        display(HTML('<div style="overflow-x:auto;max-width:100%">'+markup+'</div>'))
        if len(frame)>limit: print(f'Показано {limit} із {len(frame)} рядків.')
def run_sql(key,statement,show=True):
    if show: sql_display(statement)
    with engine.begin() as connection:
        result=connection.execute(text(statement))
        data=[dict(r) for r in result.mappings()] if result.returns_rows else []
    results[key]={'row_count':len(data),'rows':data[:20]}
    if data: table_display(data)
    else: print(key+': виконано.')
    return data

def require(name,condition,evidence=None):
    checks.append({'check':name,'passed':bool(condition),'evidence':evidence})
    assert condition,name

def load_raw(files,data_dir):
    raw_connection=engine.raw_connection()
    try:
        with raw_connection.cursor() as cursor:
            for name,f in files.items():
                table='olist_'+name+'_raw'
                command=pgsql.SQL('COPY {} FROM STDIN WITH (FORMAT CSV, HEADER TRUE, FORCE_NOT_NULL ({}))').format(pgsql.Identifier(table),pgsql.SQL(', ').join(map(pgsql.Identifier,f['columns'])))
                with (data_dir/f['archive_member']).open(encoding='utf-8',newline='') as handle:
                    cursor.copy_expert(command.as_string(cursor),handle)
        raw_connection.commit()
    except Exception: raw_connection.rollback(); raise
    finally: raw_connection.close()

def walk_plan(node,path='root'):
    yield path,node
    for i,child in enumerate(node.get('Plans',[])): yield from walk_plan(child,path+'/'+str(i))

def explain_profile(name,explain_statement,show=True):
    if show: sql_display(explain_statement)
    with engine.connect() as connection:
        lines=[r[0] for r in connection.execute(text(explain_statement))]
        json_statement=explain_statement.replace('EXPLAIN (ANALYZE, BUFFERS)','EXPLAIN (ANALYZE, BUFFERS, FORMAT JSON)',1)
        structured=connection.execute(text(json_statement)).scalar_one()[0]
    plan_text='\n'.join(lines)
    (PLAN_DIR/(name+'.txt')).write_text(plan_text+'\n')
    write_json(PLAN_DIR/(name+'.json'),structured)
    text_ms=float(re.search(r'Execution Time: ([\d.]+) ms',plan_text).group(1))
    text_planning_ms=float(re.search(r'Planning Time: ([\d.]+) ms',plan_text).group(1))
    plans[name]={'text':plan_text,'json':structured,'text_execution_ms':text_ms,'text_planning_ms':text_planning_ms}
    summary={'scenario':name,'text_execution_ms':text_ms,'text_planning_ms':text_planning_ms,'json_execution_ms':structured['Execution Time'],'actual_rows_json':structured['Plan']['Actual Rows']}
    profile_rows.append(summary)
    if show:
        display(HTML('<details open><summary>'+html.escape(name)+' · фактичний EXPLAIN (ANALYZE, BUFFERS)</summary><pre style="overflow:auto;max-height:560px">'+html.escape(plan_text)+'</pre></details>'))
    return summary

with engine.connect() as connection:
    postgres_version=connection.execute(text('SELECT version()')).scalar_one()
    server_version_num=int(connection.execute(text('SHOW server_version_num')).scalar_one())
    require('PostgreSQL 16',server_version_num//10000==16,server_version_num)
    require('Local Unix socket only',connection.execute(text('SHOW listen_addresses')).scalar_one()=='')
print(postgres_version)
print('Власний тимчасовий кластер; PostgreSQL statement_timeout=120 s. Остання клітинка зупиняє й видаляє тільки цей кластер.')

Python: 3.9.25 ; packages: {'pgserver': '0.1.4', 'psycopg2-binary': '2.9.12', 'SQLAlchemy': '2.0.54', 'pandas': '2.3.3'}


PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit
Власний тимчасовий кластер; PostgreSQL statement_timeout=120 s. Остання клітинка зупиняє й видаляє тільки цей кластер.


Усі обчислення виконуються на п'яти повних таблицях Olist. Матеріалізоване представлення зберігає повні ознаки на рівні customer_id.

## 2. Джерело, raw і typed-рівні

Використано **Brazilian E-Commerce Public Dataset by Olist**, автор Olist, [офіційна сторінка Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce), ліцензія [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). П'ять CSV взято без зміни байтів із перевіреного курсового архіву: customers — 99 441, orders — 99 441, order_items — 112 650, products — 32 951, sellers — 3 095. Закріплено API version parameter 2 та storage version 55151/2669146; оригінальний ZIP і кожен CSV ідентифіковано SHA-256. Історичний source-manifest описує також reviews попередньої теми; це ДЗ його не завантажує й не пакує. За відсутності локальних CSV notebook отримує той самий закріплений офіційний ZIP і перевіряє hashes до SQL-завантаження.

In [2]:
FILES = {'customers': {'archive_member': 'olist_customers_dataset.csv', 'sha256': '983a422239e1712ded753b3bf9ecf47dc73f144d306029dcfa99e70a226883d2', 'rows_excluding_header': 99441, 'columns': ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']}, 'orders': {'archive_member': 'olist_orders_dataset.csv', 'sha256': '8df58ef3d2d7e9944010f7beecd9b75367f5588ec6e3c91cec19ae3345ef9ecf', 'rows_excluding_header': 99441, 'columns': ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']}, 'order_items': {'archive_member': 'olist_order_items_dataset.csv', 'sha256': '0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75fab7b0e329701c3e279', 'rows_excluding_header': 112650, 'columns': ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']}, 'products': {'archive_member': 'olist_products_dataset.csv', 'sha256': '3e6569628a17fbc75fd206ee357b59e20364b9afa90f5b6cd5b4d624c58aa9cc', 'rows_excluding_header': 32951, 'columns': ['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']}, 'sellers': {'archive_member': 'olist_sellers_dataset.csv', 'sha256': '1f643d2b950373b85735e7794b20986f528d7a000432e7c6f9bcbb44d0846a0e', 'rows_excluding_header': 3095, 'columns': ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']}}
ZIP_SHA256 = '967e41e04fc306fe604e2a693f488995a8b41e5047418f8a5c8e4abd6deca784'
SOURCE_URL = 'https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce?datasetVersionNumber=2'
SOURCE_HASHES = {'work/build_notebook.py': '6d6f026547f41a26b1e4c89e15c0d7a1a57387dedcd3be67e770a8604e63da89', 'work/checks_pipeline.py': '1ea1efbb5ad14924b51c4ff0a966f21dee1dcf7a8638a7146494c19414a6001b', 'work/execute_notebook.py': '726dbeff16930ade03281456e4a310dbb26687f4d86229f8d6a2817b1e70a7a0', 'work/interpretations.json': '8c5f0bbb1ed54c3dd196b77cb8205355fff2aa67b19718ea5a549f7da3f7d568', 'work/reflection.md': 'd0841488202bd5ce3dc6152eb9df66e82434bd68217ef955f7f9003a9a12d200', 'work/runtime_helpers.py': '271bdaf86c953cd850d2b1e7643e2840865901506e0198fd08866bc1728063d8', 'work/sql/00_reset.sql': 'cc2a8ff7708ad66aa3b976e8464c1147a0d4cd8265819b25dbd2238b2f577e34', 'work/sql/01_raw_ddl.sql': 'f4c6e363b859265591bb6f23f7ba3c6907d1f65f0d64642df4d76a3188d381a8', 'work/sql/02_typed_ddl.sql': '03cfcdd09646fc565c75bf28d97042a0219633f4a12c88459e147b2e0a4fc47c', 'work/sql/03_typed_load.sql': '8e9160473489c3ac5960139b587831d091068b4a8e3960e93b5c8222edc3f501', 'work/sql/04_quality.sql': '6ed4fbbd888cb865f357a40cf858938e98a50a771f9d25204fecde0eb4c84cdb', 'work/sql/05_pipeline.sql': '75da13962ca97cb3e7c9b8f38155225372081ae24fe39715ab188b5b11880f05', 'work/sql/06_preview.sql': 'f31a4981cd8b97e44478a64a0e96939459f3e7b47f33198723a26b3589e68bd3', 'work/sql/07_stage_coverage.sql': '5533a38b13febd900ef11039c342ea6c8ce36a45eaacb8c27de1db53ffcb9be0', 'work/sql/08_referrals_setup.sql': 'ecfe02297ca35a0dea8a9443fb48136a0ee69bd19f549fa8b71d388a20a92945', 'work/sql/09_referrals_depth.sql': 'bf4f08b892d7cfe458c332cba228c5966df44a4a05af08c2e7fddbf8de031cd8', 'work/sql/10_mv_create.sql': '5e902b9234fbf0d76e2effd63d679d0c05bc8fbb7ca05a2096de08088a69174b', 'work/sql/11_mv_refresh.sql': 'a02fcb78de1126e0bc980a9984a73640897af9e62980a78fb49732adc2031296', 'work/sql/12_mv_summary.sql': '415a296fc19f6327eabd21cba6519c2524dcb73989866edcd691b694e4b6ccca', 'work/sql/13_explain_pipeline.sql': '9e58467dbd92e2eb97a2204775ab8c016781537cf435a530d58a7e38967ed467', 'work/sql/14_explain_lookup.sql': '76fbd68cbb96a3349fff6fbaa36fc74a5bb48941035c3e8b8ef51f6932ec43e4', 'work/sql/15_explain_matched.sql': 'fb4c3044c773dcec0b5d9cc1d03e41379698f8f400c023590c74a0d62edfaa9a', 'work/sql/16_mv_equivalence.sql': 'f262bec815b12cee8c2e626b2d2fe7693ab9c61c6f9095dbed1403a43cd7b651'}
DATA_DIR=PROJECT_DIR/'data/olist'
archived=course_dir/'materials/topic-04-dml-ddl/sources/olist/raw' if course_dir is not None else None
candidates=[DATA_DIR]+([archived] if archived is not None else [])
DATA_DIR=next((p for p in candidates if all((p/f['archive_member']).exists() for f in FILES.values())),DATA_DIR)
DATA_DIR.mkdir(parents=True,exist_ok=True)
if not all((DATA_DIR/f['archive_member']).exists() for f in FILES.values()):
    archive_path=DATA_DIR/'archive.zip'
    if not archive_path.exists():
        request=urllib.request.Request(SOURCE_URL,headers={'User-Agent':'Olist-course-homework/1.0'})
        with urllib.request.urlopen(request,timeout=60) as response: archive_path.write_bytes(response.read())
    require('Official pinned archive SHA-256',hashlib.sha256(archive_path.read_bytes()).hexdigest()==ZIP_SHA256)
    with zipfile.ZipFile(archive_path) as archive:
        for f in FILES.values(): (DATA_DIR/f['archive_member']).write_bytes(archive.read(f['archive_member']))
source_rows=[]
for name,f in FILES.items():
    path=DATA_DIR/f['archive_member']
    require('SHA-256: '+name,hashlib.sha256(path.read_bytes()).hexdigest()==f['sha256'])
    with path.open(encoding='utf-8',newline='') as handle:
        reader=csv.reader(handle); require('Exact CSV header: '+name,next(reader)==f['columns'])
        n_rows=sum(1 for row in reader)
    require('Logical CSV rows: '+name,n_rows==f['rows_excluding_header'],n_rows)
    source_rows.append({'table':name,'records':n_rows,'sha256':f['sha256']})
table_display(source_rows)
SQL={}

table,records,sha256
customers,99441,983a422239e1712ded753b3bf9ecf47dc73f144d306029dcfa99e70a226883d2
orders,99441,8df58ef3d2d7e9944010f7beecd9b75367f5588ec6e3c91cec19ae3345ef9ecf
order_items,112650,0bc4d068c4fe38cbb01bd90e8746e3c613fe7b4baef75fab7b0e329701c3e279
products,32951,3e6569628a17fbc75fd206ee357b59e20364b9afa90f5b6cd5b4d624c58aa9cc
sellers,3095,1f643d2b950373b85735e7794b20986f528d7a000432e7c6f9bcbb44d0846a0e


### Raw завантаження
Raw-колонки мають TEXT. `COPY CSV HEADER FORCE_NOT_NULL` зберігає вихідні порожні строки й leading zeros. Типізація відбувається окремо через `INSERT ... SELECT`: nullable порожні поля переходять у SQL NULL, числа й timestamps явно cast-яться. PostgreSQL перевіряє точні PK/FK/CHECK із DDL завдання. У raw залишено джерельні `product_*_lenght`, у typed — нормалізовані `product_*_length`.

In [3]:
SQL['00_reset'] = """
DROP SCHEMA IF EXISTS hw5 CASCADE;
CREATE SCHEMA hw5;
SET search_path TO hw5, public;
""".strip()
sql_display(SQL['00_reset']); execute_sql(SQL['00_reset']); print('00_reset: виконано.')

```sql
DROP SCHEMA IF EXISTS hw5 CASCADE;
CREATE SCHEMA hw5;
SET search_path TO hw5, public;
```

00_reset: виконано.


In [4]:
SQL['01_raw_ddl'] = """
CREATE TABLE olist_customers_raw (
    customer_id TEXT,
    customer_unique_id TEXT,
    customer_zip_code_prefix TEXT,
    customer_city TEXT,
    customer_state TEXT
);

CREATE TABLE olist_orders_raw (
    order_id TEXT,
    customer_id TEXT,
    order_status TEXT,
    order_purchase_timestamp TEXT,
    order_approved_at TEXT,
    order_delivered_carrier_date TEXT,
    order_delivered_customer_date TEXT,
    order_estimated_delivery_date TEXT
);

CREATE TABLE olist_order_items_raw (
    order_id TEXT,
    order_item_id TEXT,
    product_id TEXT,
    seller_id TEXT,
    shipping_limit_date TEXT,
    price TEXT,
    freight_value TEXT
);

CREATE TABLE olist_products_raw (
    product_id TEXT,
    product_category_name TEXT,
    product_name_lenght TEXT,
    product_description_lenght TEXT,
    product_photos_qty TEXT,
    product_weight_g TEXT,
    product_length_cm TEXT,
    product_height_cm TEXT,
    product_width_cm TEXT
);

CREATE TABLE olist_sellers_raw (
    seller_id TEXT,
    seller_zip_code_prefix TEXT,
    seller_city TEXT,
    seller_state TEXT
);
""".strip()
sql_display(SQL['01_raw_ddl']); execute_sql(SQL['01_raw_ddl']); print('01_raw_ddl: виконано.')

```sql
CREATE TABLE olist_customers_raw (
    customer_id TEXT,
    customer_unique_id TEXT,
    customer_zip_code_prefix TEXT,
    customer_city TEXT,
    customer_state TEXT
);

CREATE TABLE olist_orders_raw (
    order_id TEXT,
    customer_id TEXT,
    order_status TEXT,
    order_purchase_timestamp TEXT,
    order_approved_at TEXT,
    order_delivered_carrier_date TEXT,
    order_delivered_customer_date TEXT,
    order_estimated_delivery_date TEXT
);

CREATE TABLE olist_order_items_raw (
    order_id TEXT,
    order_item_id TEXT,
    product_id TEXT,
    seller_id TEXT,
    shipping_limit_date TEXT,
    price TEXT,
    freight_value TEXT
);

CREATE TABLE olist_products_raw (
    product_id TEXT,
    product_category_name TEXT,
    product_name_lenght TEXT,
    product_description_lenght TEXT,
    product_photos_qty TEXT,
    product_weight_g TEXT,
    product_length_cm TEXT,
    product_height_cm TEXT,
    product_width_cm TEXT
);

CREATE TABLE olist_sellers_raw (
    seller_id TEXT,
    seller_zip_code_prefix TEXT,
    seller_city TEXT,
    seller_state TEXT
);
```

01_raw_ddl: виконано.


In [5]:
load_raw(FILES,DATA_DIR)
print('COPY: п’ять повних CSV у raw TEXT; вихідні файли не змінено.')

COPY: п’ять повних CSV у raw TEXT; вихідні файли не змінено.


In [6]:
SQL['02_typed_ddl'] = """
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders      CASCADE;
DROP TABLE IF EXISTS olist_customers   CASCADE;
DROP TABLE IF EXISTS olist_products    CASCADE;
DROP TABLE IF EXISTS olist_sellers     CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);
""".strip()
sql_display(SQL['02_typed_ddl']); execute_sql(SQL['02_typed_ddl']); print('02_typed_ddl: виконано.')

```sql
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders      CASCADE;
DROP TABLE IF EXISTS olist_customers   CASCADE;
DROP TABLE IF EXISTS olist_products    CASCADE;
DROP TABLE IF EXISTS olist_sellers     CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);
```

02_typed_ddl: виконано.


In [7]:
SQL['03_typed_load'] = """
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    NULLIF(customer_zip_code_prefix, '')::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    NULLIF(order_approved_at, '')::TIMESTAMP,
    NULLIF(order_delivered_carrier_date, '')::TIMESTAMP,
    NULLIF(order_delivered_customer_date, '')::TIMESTAMP,
    NULLIF(order_estimated_delivery_date, '')::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    NULLIF(product_category_name, ''),
    NULLIF(product_name_lenght, '')::INTEGER,
    NULLIF(product_description_lenght, '')::INTEGER,
    NULLIF(product_photos_qty, '')::INTEGER,
    NULLIF(product_weight_g, '')::INTEGER,
    NULLIF(product_length_cm, '')::INTEGER,
    NULLIF(product_height_cm, '')::INTEGER,
    NULLIF(product_width_cm, '')::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    NULLIF(seller_zip_code_prefix, '')::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    NULLIF(product_id, ''),
    NULLIF(seller_id, ''),
    NULLIF(shipping_limit_date, '')::TIMESTAMP,
    NULLIF(price, '')::NUMERIC(12, 2),
    NULLIF(freight_value, '')::NUMERIC(12, 2)
FROM olist_order_items_raw;
""".strip()
sql_display(SQL['03_typed_load']); execute_sql(SQL['03_typed_load']); print('03_typed_load: виконано.')

```sql
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    NULLIF(customer_zip_code_prefix, '')::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    NULLIF(order_approved_at, '')::TIMESTAMP,
    NULLIF(order_delivered_carrier_date, '')::TIMESTAMP,
    NULLIF(order_delivered_customer_date, '')::TIMESTAMP,
    NULLIF(order_estimated_delivery_date, '')::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    NULLIF(product_category_name, ''),
    NULLIF(product_name_lenght, '')::INTEGER,
    NULLIF(product_description_lenght, '')::INTEGER,
    NULLIF(product_photos_qty, '')::INTEGER,
    NULLIF(product_weight_g, '')::INTEGER,
    NULLIF(product_length_cm, '')::INTEGER,
    NULLIF(product_height_cm, '')::INTEGER,
    NULLIF(product_width_cm, '')::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    NULLIF(seller_zip_code_prefix, '')::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    NULLIF(product_id, ''),
    NULLIF(seller_id, ''),
    NULLIF(shipping_limit_date, '')::TIMESTAMP,
    NULLIF(price, '')::NUMERIC(12, 2),
    NULLIF(freight_value, '')::NUMERIC(12, 2)
FROM olist_order_items_raw;
```

03_typed_load: виконано.


In [8]:
execute_sql('ANALYZE')
"""Meaningful invariant and boundary checks, embedded in the notebook by the builder."""
# All fixtures run in transactions that are rolled back; original Olist rows remain intact.
def validate_raw_typed():
    counts=[]
    for name,f in FILES.items():
        raw_n=scalar('SELECT COUNT(*) FROM olist_'+name+'_raw')
        typed_n=scalar('SELECT COUNT(*) FROM olist_'+name)
        require('raw / typed / original rows: '+name,raw_n==typed_n==f['rows_excluding_header'],[raw_n,typed_n])
        counts.append({'table':name,'raw':raw_n,'typed':typed_n,'source':f['rows_excluding_header']})
    require('Every raw field is TEXT',scalar("SELECT COUNT(*) FROM information_schema.columns WHERE table_schema='hw5' AND table_name LIKE '%_raw' AND data_type <> 'text'")==0)
    # A leading-zero postal code must survive in raw and become an integer in typed.
    postal=rows("SELECT r.customer_zip_code_prefix AS raw_zip,c.customer_zip_code_prefix AS typed_zip FROM olist_customers_raw r JOIN olist_customers c USING(customer_id) WHERE r.customer_zip_code_prefix LIKE '0%' ORDER BY r.customer_id LIMIT 1")[0]
    require('Raw leading zero preserved',postal['raw_zip'].startswith('0') and int(postal['raw_zip'])==postal['typed_zip'],postal)
    require('Raw blank / typed NULL nullable timestamp',scalar("SELECT COUNT(*) FROM olist_orders_raw r JOIN olist_orders t USING(order_id) WHERE r.order_approved_at='' AND t.order_approved_at IS NULL")==scalar("SELECT COUNT(*) FROM olist_orders_raw WHERE order_approved_at=''"))
    results['raw_typed_counts']={'rows':counts}; table_display(counts)


def validate_features():
    quality=rows(SQL['04_quality'])[0]
    summary=rows(SQL['12_mv_summary'])[0]
    require('All customer keys in MV',summary['n_rows']==summary['n_keys']==quality['customers'],summary)
    require('SUM frequency equals all-status order count',summary['orders']==quality['orders'])
    require('SUM monetary reconciles to raw items plus freight',summary['monetary_brl']==quality['item_value_plus_freight_brl'])
    require('Olist customer_id grain is single-order',summary['frequency_not_one']==0)
    require('Real buyer identity is a different grain',quality['real_buyers']<quality['customers'])
    require('Recent AOV = one order total on this dataset',scalar('SELECT COUNT(*) FROM mv_olist_customer_features WHERE avg_order_value_30d_recent <> monetary')==0)
    require('Recency correct for every customer',scalar("SELECT COUNT(*) FROM mv_olist_customer_features m JOIN olist_orders o USING(customer_id) WHERE m.recency_days <> (DATE '"+str(quality['reference_date'])+"'-o.order_purchase_timestamp::DATE)")==0)
    require('Cohort quarter correct for every customer',scalar("SELECT COUNT(*) FROM mv_olist_customer_features WHERE EXTRACT(MONTH FROM cohort_quarter) NOT IN (1,4,7,10) OR EXTRACT(DAY FROM cohort_quarter)<>1 OR EXTRACT(YEAR FROM cohort_quarter)<>EXTRACT(YEAR FROM first_order_date) OR EXTRACT(QUARTER FROM cohort_quarter)<>EXTRACT(QUARTER FROM first_order_date)")==0)
    require('Cancellation denominator all orders',scalar('SELECT SUM(cancellation_rate) FROM mv_olist_customer_features')==Decimal(quality['canceled_orders']))
    require('Category revenue bounded by monetary',scalar('SELECT COUNT(*) FROM mv_olist_customer_features WHERE preferred_category_revenue > monetary OR preferred_category_revenue < 0')==0)
    require('Itemless orders retained as zero feature records',summary['no_item_customers']==quality['orders_without_items'])
    require('Full pipeline equals full MV before refresh',scalar(SQL['16_mv_equivalence'])==0)
    index=rows("SELECT i.indisunique,i.indisvalid,i.indpred IS NULL AS not_partial,i.indexprs IS NULL AS not_expression,pg_get_indexdef(i.indexrelid) AS definition FROM pg_index i WHERE i.indexrelid='idx_mv_olist_customer_features_customer_id'::regclass")[0]
    require('Valid full UNIQUE index on customer_id',all(index[k] for k in ['indisunique','indisvalid','not_partial','not_expression']) and '(customer_id)' in index['definition'],index)
    results['feature_summary']={'rows':[summary]}; results['quality']={'rows':[quality]}
    table_display([summary])


def validate_stages_and_lookup():
    stage_rows=rows(SQL['07_stage_coverage'])
    counts={r['stage']:r['n_rows'] for r in stage_rows}
    expected={'reference_date':1, **{name:99441 for name in ['stage1_orders_total','stage2_rfm','stage3_cohort','stage4_rolling_aov','stage5_cancel_rate']}, 'stage6_category_revenue':99470,'stage7_preferred_category':98666}
    require('All eight named stages have expected independent source counts',counts==expected,counts)
    independent_category_pairs=scalar("SELECT COUNT(*) FROM (SELECT o.customer_id,COALESCE(p.product_category_name,'(unknown)') FROM olist_orders o JOIN olist_order_items i USING(order_id) LEFT JOIN olist_products p USING(product_id) GROUP BY 1,2) x")
    require('Category-stage grain matches source customer × category',counts['stage6_category_revenue']==independent_category_pairs)
    mv_lookup=rows(SQL['14_explain_lookup'].split('\n',1)[1])
    live_lookup=rows(SQL['15_explain_matched'].split('\n',1)[1])
    require('Matched pipeline / MV lookup return identical customer and 11 columns',len(mv_lookup)==len(live_lookup)==1 and mv_lookup==live_lookup and len(mv_lookup[0])==11,mv_lookup)
    results['matched_lookup_result']={'rows':mv_lookup}


def validate_pipeline_boundaries():
    connection=engine.connect(); transaction=connection.begin()
    try:
        connection.execute(text("SET LOCAL statement_timeout='10s'"))
        connection.execute(text("INSERT INTO olist_customers(customer_id,customer_unique_id,customer_state) VALUES ('hw5_test_a','hw5_test_a','SP'),('hw5_test_none','hw5_test_none','RJ'),('hw5_test_null','hw5_test_null','MG'),('hw5_test_tie','hw5_test_tie','BA')"))
        connection.execute(text("INSERT INTO olist_products(product_id,product_category_name) VALUES ('hw5_test_alpha','alpha'),('hw5_test_beta','beta')"))
        connection.execute(text("""INSERT INTO olist_orders(order_id,customer_id,order_status,order_purchase_timestamp) VALUES
        ('hw5_test_o1','hw5_test_a','delivered','2018-01-01 12:00'),
        ('hw5_test_o2','hw5_test_a','canceled','2018-01-31 12:00'),
        ('hw5_test_o3','hw5_test_a','shipped','2018-01-31 12:00'),
        ('hw5_test_o4','hw5_test_a','delivered','2018-02-01 12:00'),
        ('hw5_test_o_null','hw5_test_null','unavailable','2018-01-02'),
        ('hw5_test_o_tie','hw5_test_tie','delivered','2018-01-03');"""))
        connection.execute(text("""INSERT INTO olist_order_items(order_id,order_item_id,product_id,price,freight_value) VALUES
        ('hw5_test_o1',1,'hw5_test_alpha',8,2),('hw5_test_o2',1,'hw5_test_alpha',18,2),
        ('hw5_test_o3',1,'hw5_test_alpha',28,2),('hw5_test_o4',1,'hw5_test_alpha',38,2),
        ('hw5_test_o_null',1,NULL,NULL,5),('hw5_test_o_null',2,NULL,7,NULL),
        ('hw5_test_o_tie',1,'hw5_test_beta',9,1),('hw5_test_o_tie',2,'hw5_test_alpha',8,2);"""))
        ctes=SQL['05_pipeline'][:SQL['05_pipeline'].index('\nSELECT\n    c.customer_id,')]
        window=rows(ctes+"\nSELECT * FROM stage4_rolling_aov WHERE customer_id='hw5_test_a' ORDER BY order_ts,order_id",connection)
        require('30-day inclusive lower boundary + timestamp peers',[(r['order_id'],r['avg_order_value_30d']) for r in window]==[('hw5_test_o1',Decimal('10.00')),('hw5_test_o2',Decimal('20.00')),('hw5_test_o3',Decimal('20.00')),('hw5_test_o4',Decimal('30.00'))],window)
        require('Stable latest timestamp + order_id',next(r for r in window if r['rn_recent']==1)['order_id']=='hw5_test_o4')
        peer_latest=rows(ctes+"\nSELECT order_id,rn_recent FROM stage4_rolling_aov WHERE customer_id='hw5_test_a' AND order_ts='2018-01-31 12:00' ORDER BY rn_recent",connection)
        require('Peer latest tie resolved descending order_id',peer_latest[0]['order_id']=='hw5_test_o3')
        features=rows(SQL['05_pipeline'].rstrip(';')+"\nWHERE c.customer_id LIKE 'hw5_test_%' ORDER BY c.customer_id",connection)
        features={r['customer_id']:r for r in features}
        a=features['hw5_test_a']; none=features['hw5_test_none']; null=features['hw5_test_null']; tie=features['hw5_test_tie']
        require('Fixture all statuses / freight / no fanout',a['frequency']==4 and a['monetary']==Decimal('100.00') and a['cancellation_rate']==Decimal('0.2500'),a)
        require('Fixture latest 30d AOV',a['avg_order_value_30d_recent']==Decimal('30.00'))
        require('Fixture no orders sentinel + NULL cohort',none['frequency']==0 and none['recency_days']==9999 and none['monetary']==0 and none['first_order_date'] is None and none['cohort_quarter'] is None and none['preferred_category']=='(none)',none)
        require('Fixture nullable components / category',null['monetary']==Decimal('12.00') and null['preferred_category']=='(unknown)' and null['cancellation_rate']==0,null)
        require('Fixture top category deterministic tie',tie['preferred_category']=='alpha' and tie['preferred_category_revenue']==Decimal('10.00'),tie)
        results['boundary_fixture']={'rows':list(features.values())}; table_display(list(features.values()))
    finally: transaction.rollback(); connection.close()
    require('Pipeline fixtures rolled back',scalar("SELECT COUNT(*) FROM olist_customers WHERE customer_id LIKE 'hw5_test_%'")==0)


def validate_recursion():
    ctes=SQL['09_referrals_depth'][:SQL['09_referrals_depth'].index('\nSELECT\n    depth,')]
    full=rows(ctes+'\nSELECT customer_id,depth,path_ids FROM referral_depth ORDER BY depth,customer_id')
    require('Synthetic referrals exactly 9 edges',scalar('SELECT COUNT(*) FROM hw5_referrals')==9)
    require('Root chain + isolated customers cover all customers',len(full)==scalar('SELECT COUNT(*) FROM olist_customers') and len({r['customer_id'] for r in full})==len(full))
    require('Referral chain reaches depth 10',max(r['depth'] for r in full)==10)
    require('Every path depth matches length and no repeated ids',all(r['depth']==len(r['path_ids'])==len(set(r['path_ids'])) for r in full))
    depth_counts=rows(SQL['09_referrals_depth'])
    require('Isolated customers are included as roots',depth_counts[0]['n_customers']==99432 and all(r['n_customers']==1 for r in depth_counts[1:]),depth_counts)
    # Small fixtures preserve the same anchor/recursive expression but use temporary tables.
    fixture_ctes=ctes.replace('olist_customers','hw5_test_customers').replace('hw5_referrals','hw5_test_referrals')
    connection=engine.connect(); transaction=connection.begin()
    try:
        connection.execute(text("SET LOCAL statement_timeout='3s'"))
        connection.execute(text("CREATE TEMP TABLE hw5_test_customers(customer_id TEXT PRIMARY KEY); CREATE TEMP TABLE hw5_test_referrals(referrer_customer_id TEXT NOT NULL,referred_customer_id TEXT PRIMARY KEY)"))
        connection.execute(text("INSERT INTO hw5_test_customers VALUES ('A'),('B'),('R'),('C'),('I'); INSERT INTO hw5_test_referrals VALUES ('A','B'),('B','A'),('R','C')"))
        rooted=rows(fixture_ctes+'\nSELECT customer_id,depth,path_ids FROM referral_depth ORDER BY customer_id',connection)
        require('Root traversal terminates but omits disconnected A-B cycle',{r['customer_id'] for r in rooted}=={'R','C','I'},rooted)
        allseed=fixture_ctes[:fixture_ctes.index('\n    WHERE NOT EXISTS (')]+fixture_ctes[fixture_ctes.index('\n\n    UNION ALL'):]
        reachable=rows(allseed+'\nSELECT customer_id,depth,path_ids FROM referral_depth ORDER BY customer_id,depth',connection)
        require('All-node anchors + visited path terminate A-B cycle',len(reachable)==8 and max(r['depth'] for r in reachable)==2 and {r['customer_id'] for r in reachable}=={'A','B','R','C','I'},reachable)
        connection.execute(text("TRUNCATE hw5_test_customers,hw5_test_referrals; INSERT INTO hw5_test_customers SELECT 'N'||LPAD(n::TEXT,2,'0') FROM generate_series(1,55) n; INSERT INTO hw5_test_referrals SELECT 'N'||LPAD(n::TEXT,2,'0'),'N'||LPAD((n+1)::TEXT,2,'0') FROM generate_series(1,54) n"))
        long_chain=rows(fixture_ctes+'\nSELECT customer_id,depth,path_ids FROM referral_depth ORDER BY depth',connection)
        require('Depth guard truncates long chain: 50 of 55 nodes',len(long_chain)==50 and max(r['depth'] for r in long_chain)==50,long_chain[-1])
        results['recursion_fixtures']={'rooted_cycle_rows':rooted,'all_seed_rows':reachable,'long_chain_rows':len(long_chain),'long_chain_max_depth':50,'omitted_long_chain_nodes':5}
        print('Цикл A ↔ B: root-only traversal повернув R, C, I; A/B пропущені. All-node seed повернув 8 path-рядків, depth ≤ 2. Ланцюг із 55 вузлів обрізано до depth=50.')
    finally: transaction.rollback(); connection.close()


def validate_constraints():
    cases=[
      ('Duplicate customer key','23505',"INSERT INTO olist_customers SELECT * FROM olist_customers LIMIT 1"),
      ('Invalid order status','23514',"UPDATE olist_orders SET order_status='bad' WHERE order_id=(SELECT MIN(order_id) FROM olist_orders)"),
      ('Negative item price','23514',"UPDATE olist_order_items SET price=-1 WHERE (order_id,order_item_id)=(SELECT order_id,order_item_id FROM olist_order_items ORDER BY order_id,order_item_id LIMIT 1)"),
      ('Missing customer foreign key','23503',"INSERT INTO olist_orders(order_id,customer_id,order_status,order_purchase_timestamp) VALUES ('hw5_test_bad','missing_customer','delivered','2018-01-01')"),
      ('Self-referral','23514',"INSERT INTO hw5_referrals(referrer_customer_id,referred_customer_id) SELECT MIN(customer_id),MIN(customer_id) FROM olist_customers"),
    ]
    observed=[]
    for name,expected,statement in cases:
        connection=engine.connect(); transaction=connection.begin()
        try:
            connection.execute(text("SET LOCAL statement_timeout='3s'"))
            try: connection.execute(text(statement)); actual=None
            except Exception as error: actual=getattr(getattr(error,'orig',error),'pgcode',None)
            require(name+' rejected',actual==expected,actual)
            observed.append({'fixture':name,'expected_sqlstate':expected,'actual_sqlstate':actual})
        finally: transaction.rollback(); connection.close()
    connection=engine.connect(); transaction=connection.begin()
    try:
        connection.execute(text("SET LOCAL statement_timeout='3s'; CREATE MATERIALIZED VIEW hw5_test_no_unique AS SELECT 1 AS id"))
        try: connection.execute(text('REFRESH MATERIALIZED VIEW CONCURRENTLY hw5_test_no_unique')); actual=None
        except Exception as error: actual=getattr(getattr(error,'orig',error),'pgcode',None)
        require('CONCURRENTLY requires full unique index',actual=='55000',actual)
        observed.append({'fixture':'MV missing UNIQUE index','expected_sqlstate':'55000','actual_sqlstate':actual})
    finally: transaction.rollback(); connection.close()
    results['negative_constraints']={'rows':observed}; table_display(observed)


def validate_refresh_change():
    # Exercise staleness, actual refresh, and rollback in one transaction on the owned database.
    connection=engine.connect(); transaction=connection.begin()
    try:
        connection.execute(text("SET LOCAL statement_timeout='30s'"))
        item=rows('SELECT i.order_id,i.order_item_id,o.customer_id FROM olist_order_items i JOIN olist_orders o USING(order_id) ORDER BY i.order_id,i.order_item_id LIMIT 1',connection)[0]
        identifier=item['customer_id']
        before=scalar("SELECT monetary FROM mv_olist_customer_features WHERE customer_id='"+identifier+"'",connection)
        connection.execute(text('UPDATE olist_order_items SET price=price+1 WHERE order_id=:order_id AND order_item_id=:order_item_id'),item)
        stale=scalar("SELECT monetary FROM mv_olist_customer_features WHERE customer_id='"+identifier+"'",connection)
        live=scalar("SELECT SUM(COALESCE(i.price,0)+COALESCE(i.freight_value,0)) FROM olist_orders o JOIN olist_order_items i USING(order_id) WHERE o.customer_id='"+identifier+"'",connection)
        require('MV remains stale after base-table change',stale==before and live==before+Decimal('1.00'))
        connection.execute(text(SQL['11_mv_refresh']))
        refreshed=scalar("SELECT monetary FROM mv_olist_customer_features WHERE customer_id='"+identifier+"'",connection)
        require('CONCURRENT refresh incorporates changed source',refreshed==live)
        evidence={'customer_id':identifier,'before_brl':before,'stale_brl':stale,'changed_source_brl':live,'refreshed_brl':refreshed}
        results['refresh_fixture']={'rows':[evidence]}; table_display([evidence])
    finally: transaction.rollback(); connection.close()
    require('Rollback restores source and MV equality',scalar(SQL['16_mv_equivalence'])==0)
    print('Транзакцію відкочено; вихідні дані та початковий MV відновлено.')

validate_raw_typed()

table,raw,typed,source
customers,99441,99441,99441
orders,99441,99441,99441
order_items,112650,112650,112650
products,32951,32951,32951
sellers,3095,3095,3095


In [9]:
SQL['04_quality'] = """
SELECT
    (SELECT COUNT(*) FROM olist_customers) AS customers,
    (SELECT COUNT(DISTINCT customer_unique_id) FROM olist_customers) AS real_buyers,
    (SELECT COUNT(*) FROM olist_orders) AS orders,
    (SELECT COUNT(*) FROM olist_order_items) AS order_items,
    (SELECT COUNT(*) FROM olist_products) AS products,
    (SELECT COUNT(*) FROM olist_sellers) AS sellers,
    (SELECT (MAX(order_purchase_timestamp)::DATE+1) FROM olist_orders) AS reference_date,
    (SELECT SUM(COALESCE(price,0)+COALESCE(freight_value,0)) FROM olist_order_items) AS item_value_plus_freight_brl,
    (SELECT COUNT(*) FROM olist_orders WHERE order_status='canceled') AS canceled_orders,
    (SELECT COUNT(*) FROM olist_orders AS o WHERE NOT EXISTS (SELECT 1 FROM olist_order_items AS i WHERE i.order_id=o.order_id)) AS orders_without_items,
    (SELECT COUNT(*) FROM olist_products WHERE product_category_name IS NULL) AS products_without_category;
""".strip()
run_sql('04_quality', SQL['04_quality'])

```sql
SELECT
    (SELECT COUNT(*) FROM olist_customers) AS customers,
    (SELECT COUNT(DISTINCT customer_unique_id) FROM olist_customers) AS real_buyers,
    (SELECT COUNT(*) FROM olist_orders) AS orders,
    (SELECT COUNT(*) FROM olist_order_items) AS order_items,
    (SELECT COUNT(*) FROM olist_products) AS products,
    (SELECT COUNT(*) FROM olist_sellers) AS sellers,
    (SELECT (MAX(order_purchase_timestamp)::DATE+1) FROM olist_orders) AS reference_date,
    (SELECT SUM(COALESCE(price,0)+COALESCE(freight_value,0)) FROM olist_order_items) AS item_value_plus_freight_brl,
    (SELECT COUNT(*) FROM olist_orders WHERE order_status='canceled') AS canceled_orders,
    (SELECT COUNT(*) FROM olist_orders AS o WHERE NOT EXISTS (SELECT 1 FROM olist_order_items AS i WHERE i.order_id=o.order_id)) AS orders_without_items,
    (SELECT COUNT(*) FROM olist_products WHERE product_category_name IS NULL) AS products_without_category;
```

,value
metric,
customers,99441
real_buyers,96096
orders,99441
order_items,112650
products,32951
sellers,3095
reference_date,2018-10-18
item_value_plus_freight_brl,15843553.24
canceled_orders,625


[{'customers': 99441,
  'real_buyers': 96096,
  'orders': 99441,
  'order_items': 112650,
  'products': 32951,
  'sellers': 3095,
  'reference_date': datetime.date(2018, 10, 18),
  'item_value_plus_freight_brl': Decimal('15843553.24'),
  'canceled_orders': 625,
  'orders_without_items': 775,
  'products_without_category': 610}]

Raw-таблиці зберігають вихідні поля як TEXT; typed-таблиці виконують явні перетворення та перевіряють PK/FK/CHECK. Порожні nullable-поля переходять у SQL NULL, а CSV-назви product_*_lenght нормалізуються до product_*_length. Перевірка counts не дозволяє мовчки втратити рядки. Рівень фінальної ознаки за рубрикою — **customer_id**, тобто 99 441 ключ; реальний повторний покупець має customer_unique_id, яких 96 096. У цій копії frequency на customer_id дорівнює 1; багатозамовна логіка перевіряється окремими транзакційними fixtures. Контрольна дата 2018-10-18 є MAX purchase date + 1, а не поточною датою комп'ютера.

## 3. Повний CTE-chain із восьми named stages
0: reference_date; 1: order totals; 2: RFM; 3: cohort quarter; 4: 30-day AOV та deterministic latest row; 5: cancellation rate; 6: category revenue; 7: preferred category. Кожний stage має `-- Stage N` і окрему логічну роль.

Завдання показує LIMIT 20 для preview; canonical SQL не містить LIMIT і повертає всі customer_id. До MV та full EXPLAIN переходить саме ця версія. Повний pipeline має одинадцять колонок; preview нижче обмежує лише показ.

In [10]:
SQL['05_pipeline'] = """
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id;
""".strip()
PIPELINE_SQL=SQL['05_pipeline']
PIPELINE_HASH=hashlib.sha256(PIPELINE_SQL.strip().encode()).hexdigest()
sql_display(PIPELINE_SQL)
print('Canonical pipeline SHA-256:',PIPELINE_HASH)

```sql
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id;
```

Canonical pipeline SHA-256: 69915687e8d2548749a33502f0fa9b1fee9f147c393dce10c163323b06132f91


In [11]:
SQL['06_preview'] = """
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 10;
""".strip()
run_sql('06_preview', SQL['06_preview'])

```sql
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 10;
```

customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0000,telefonia_fixa,13664.08
ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0000,telefonia_fixa,7274.88
c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0000,utilidades_domesticas,6929.31
f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0000,pcs,6922.21
3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0000,artes,6726.66
05455dfa7cd02f13d132aa7a6a9729c6,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0000,agro_industria_e_comercio,6081.54
df55c14d1476a9a3467f131269c2477f,RJ,565,1,4950.34,2017-04-01,2017-04-01,4950.34,0.0000,eletroportateis,4950.34
e0a2412720e9ea4f26c1ac985f6a7358,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0000,cool_stuff,4809.44
24bbf5fd2f2e1b359ee7de94defc4a15,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0000,eletroportateis,4764.34
3d979689f636322c62418b6346b1c6d2,PB,118,1,4681.78,2018-06-22,2018-04-01,4681.78,0.0000,pcs,4681.78


[{'customer_id': '1617b1357756262bfa56ab541c47bc16',
  'customer_state': 'RJ',
  'recency_days': 384,
  'frequency': 1,
  'monetary': Decimal('13664.08'),
  'first_order_date': datetime.date(2017, 9, 29),
  'cohort_quarter': datetime.date(2017, 7, 1),
  'avg_order_value_30d_recent': Decimal('13664.08'),
  'cancellation_rate': Decimal('0.0000'),
  'preferred_category': 'telefonia_fixa',
  'preferred_category_revenue': Decimal('13664.08')},
 {'customer_id': 'ec5b2ba62e574342386871631fafd3fc',
  'customer_state': 'ES',
  'recency_days': 95,
  'frequency': 1,
  'monetary': Decimal('7274.88'),
  'first_order_date': datetime.date(2018, 7, 15),
  'cohort_quarter': datetime.date(2018, 7, 1),
  'avg_order_value_30d_recent': Decimal('7274.88'),
  'cancellation_rate': Decimal('0.0000'),
  'preferred_category': 'telefonia_fixa',
  'preferred_category_revenue': Decimal('7274.88')},
 {'customer_id': 'c6e2731c5b391845f6800c97401a43a9',
  'customer_state': 'MS',
  'recency_days': 613,
  'frequency': 1

In [12]:
SQL['07_stage_coverage'] = """
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT 'reference_date' AS stage, COUNT(*) AS n_rows FROM reference_date
UNION ALL SELECT 'stage1_orders_total', COUNT(*) FROM stage1_orders_total
UNION ALL SELECT 'stage2_rfm', COUNT(*) FROM stage2_rfm
UNION ALL SELECT 'stage3_cohort', COUNT(*) FROM stage3_cohort
UNION ALL SELECT 'stage4_rolling_aov', COUNT(*) FROM stage4_rolling_aov
UNION ALL SELECT 'stage5_cancel_rate', COUNT(*) FROM stage5_cancel_rate
UNION ALL SELECT 'stage6_category_revenue', COUNT(*) FROM stage6_category_revenue
UNION ALL SELECT 'stage7_preferred_category', COUNT(*) FROM stage7_preferred_category;
""".strip()
run_sql('07_stage_coverage', SQL['07_stage_coverage'])

```sql
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT 'reference_date' AS stage, COUNT(*) AS n_rows FROM reference_date
UNION ALL SELECT 'stage1_orders_total', COUNT(*) FROM stage1_orders_total
UNION ALL SELECT 'stage2_rfm', COUNT(*) FROM stage2_rfm
UNION ALL SELECT 'stage3_cohort', COUNT(*) FROM stage3_cohort
UNION ALL SELECT 'stage4_rolling_aov', COUNT(*) FROM stage4_rolling_aov
UNION ALL SELECT 'stage5_cancel_rate', COUNT(*) FROM stage5_cancel_rate
UNION ALL SELECT 'stage6_category_revenue', COUNT(*) FROM stage6_category_revenue
UNION ALL SELECT 'stage7_preferred_category', COUNT(*) FROM stage7_preferred_category;
```

stage,n_rows
reference_date,1
stage1_orders_total,99441
stage2_rfm,99441
stage3_cohort,99441
stage4_rolling_aov,99441
stage5_cancel_rate,99441
stage6_category_revenue,99470
stage7_preferred_category,98666


[{'stage': 'reference_date', 'n_rows': 1},
 {'stage': 'stage1_orders_total', 'n_rows': 99441},
 {'stage': 'stage2_rfm', 'n_rows': 99441},
 {'stage': 'stage3_cohort', 'n_rows': 99441},
 {'stage': 'stage4_rolling_aov', 'n_rows': 99441},
 {'stage': 'stage5_cancel_rate', 'n_rows': 99441},
 {'stage': 'stage6_category_revenue', 'n_rows': 99470},
 {'stage': 'stage7_preferred_category', 'n_rows': 98666}]

Ланцюжок має reference_date та **сім named stages** із коментарями: order totals → RFM → cohort quarter; окремі гілки обчислюють rolling AOV, cancellation rate та category revenue → preferred category. Stage 1 спершу повертає один рядок на order, тому item JOIN не збільшує frequency. LEFT JOIN зберігає замовлення без товарних позицій. Monetary і category revenue враховують усі статуси та freight: це записана сума вартості, а не прибуток чи підтверджений чистий дохід.

RANGE 30 days включає точну нижню межу та всіх peers із тим самим order_ts. avg_order_value_30d_recent береться на останньому замовленні customer_id; це не окреме вікно від кінцевої reference_date. ROW_NUMBER обирає останній рядок за order_ts DESC, order_id DESC; категорію з найбільшою сумою — за revenue DESC, потім назвою за collation PostgreSQL. NULL category має label (unknown), відсутність позицій — (none). Final SELECT починається з усіх customers; preview LIMIT використовується лише для показу, повний pipeline і MV не обмежені. Це профіль стану наприкінці датасету; для point-in-time навчання додатково потрібні часовий cutoff і контроль доступності подій та статусів на дату прогнозу.

## 4. Referral chain: anchor, recursion, path і depth

In [13]:
SQL['08_referrals_setup'] = """
DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH ordered_customers AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT
    prev.customer_id AS referrer_customer_id,
    nxt.customer_id AS referred_customer_id
FROM ordered_customers AS prev
JOIN ordered_customers AS nxt
    ON nxt.rn = prev.rn + 1
ON CONFLICT (referred_customer_id) DO NOTHING;
""".strip()
sql_display(SQL['08_referrals_setup']); execute_sql(SQL['08_referrals_setup']); print('08_referrals_setup: виконано.')

```sql
DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH ordered_customers AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT
    prev.customer_id AS referrer_customer_id,
    nxt.customer_id AS referred_customer_id
FROM ordered_customers AS prev
JOIN ordered_customers AS nxt
    ON nxt.rn = prev.rn + 1
ON CONFLICT (referred_customer_id) DO NOTHING;
```

08_referrals_setup: виконано.


In [14]:
run_sql('referral_edges','SELECT * FROM hw5_referrals ORDER BY referrer_customer_id,referred_customer_id')

```sql
SELECT * FROM hw5_referrals ORDER BY referrer_customer_id,referred_customer_id
```

referrer_customer_id,referred_customer_id,referred_at
00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140,2026-10-17 02:16:11.137141+04:00
000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079,2026-10-17 02:16:11.137141+04:00
0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5,2026-10-17 02:16:11.137141+04:00
0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb,2026-10-17 02:16:11.137141+04:00
000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652,2026-10-17 02:16:11.137141+04:00
0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086,2026-10-17 02:16:11.137141+04:00
000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282,2026-10-17 02:16:11.137141+04:00
00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c,2026-10-17 02:16:11.137141+04:00
00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130,2026-10-17 02:16:11.137141+04:00


[{'referrer_customer_id': '00012a2ce6f8dcda20d059ce98491703',
  'referred_customer_id': '000161a058600d5901f007fab4c27140',
  'referred_at': datetime.datetime(2026, 10, 17, 2, 16, 11, 137141, tzinfo=datetime.timezone(datetime.timedelta(seconds=14400)))},
 {'referrer_customer_id': '000161a058600d5901f007fab4c27140',
  'referred_customer_id': '0001fd6190edaaf884bcaf3d49edf079',
  'referred_at': datetime.datetime(2026, 10, 17, 2, 16, 11, 137141, tzinfo=datetime.timezone(datetime.timedelta(seconds=14400)))},
 {'referrer_customer_id': '0001fd6190edaaf884bcaf3d49edf079',
  'referred_customer_id': '0002414f95344307404f0ace7a26f1d5',
  'referred_at': datetime.datetime(2026, 10, 17, 2, 16, 11, 137141, tzinfo=datetime.timezone(datetime.timedelta(seconds=14400)))},
 {'referrer_customer_id': '0002414f95344307404f0ace7a26f1d5',
  'referred_customer_id': '000379cdec625522490c315e70c7a9fb',
  'referred_at': datetime.datetime(2026, 10, 17, 2, 16, 11, 137141, tzinfo=datetime.timezone(datetime.timedelta

In [15]:
SQL['09_referrals_depth'] = """
WITH RECURSIVE referral_depth AS (
    -- ANCHOR: root customers, які не були referred іншими customers
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: перейти від referrer до його referred customers
    SELECT
        r.referred_customer_id AS customer_id,
        rd.depth + 1 AS depth,
        rd.path_ids || r.referred_customer_id AS path_ids,
        rd.path_text || ' -> ' || r.referred_customer_id AS path_text
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON rd.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(rd.path_ids)
      AND rd.depth < 50
)
SELECT
    depth,
    COUNT(*) AS n_customers,
    MAX(path_text) AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;
""".strip()
run_sql('09_referrals_depth', SQL['09_referrals_depth'])

```sql
WITH RECURSIVE referral_depth AS (
    -- ANCHOR: root customers, які не були referred іншими customers
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: перейти від referrer до його referred customers
    SELECT
        r.referred_customer_id AS customer_id,
        rd.depth + 1 AS depth,
        rd.path_ids || r.referred_customer_id AS path_ids,
        rd.path_text || ' -> ' || r.referred_customer_id AS path_text
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON rd.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(rd.path_ids)
      AND rd.depth < 50
)
SELECT
    depth,
    COUNT(*) AS n_customers,
    MAX(path_text) AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;
```

depth,n_customers,sample_path
1,99432,ffffe8b65bbe3087b653a978c870db99
2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140
3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079
4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5
5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb
6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652
7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086
8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282
9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282 -> 00050bf6e01e69d5c0fd612f1bcfb69c
10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282 -> 00050bf6e01e69d5c0fd612f1bcfb69c -> 000598caf2ef4117407665ac33275130


[{'depth': 1,
  'n_customers': 99432,
  'sample_path': 'ffffe8b65bbe3087b653a978c870db99'},
 {'depth': 2,
  'n_customers': 1,
  'sample_path': '00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140'},
 {'depth': 3,
  'n_customers': 1,
  'sample_path': '00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079'},
 {'depth': 4,
  'n_customers': 1,
  'sample_path': '00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5'},
 {'depth': 5,
  'n_customers': 1,
  'sample_path': '00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb'},
 {'depth': 6,
  'n_customers': 1,
  'sample_path': '00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 

Olist не містить реальних referral-зв'язків. Синтетична hw5_referrals з'єднує перші десять відсортованих customer ID у дев'ять ребер; інші customers лишаються ізольованими коренями. Anchor використовує NULL-safe NOT EXISTS, recursive part додає depth і path, перевірка path_ids забороняє повторний вузол, depth < 50 обмежує обхід. Depth 1 означає початковий customer, а не кількість ребер.

Завершення та охоплення графа — різні властивості. Цикл A → B → A без кореня не потрапить до цього anchor, отже A/B будуть пропущені. При старті з циклу перевірка visited path зупинить повторний A. Fixtures показують обидва сценарії та відсікання коректного ланцюга з 55 вузлів на depth 50. Для production потрібна глобальна валідація циклів або інша схема стартових вузлів.

## 5. Materialized View повного pipeline

In [16]:
SQL['10_mv_create'] = """
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

CREATE MATERIALIZED VIEW mv_olist_customer_features AS
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id;

CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features (customer_id);
""".strip()
require('MV reuses the entire canonical query',SQL['10_mv_create'].split('CREATE MATERIALIZED VIEW mv_olist_customer_features AS\n',1)[1].split(';\n\nCREATE UNIQUE INDEX',1)[0].strip()==PIPELINE_SQL.rstrip(';').strip())
sql_display(SQL['10_mv_create']); execute_sql(SQL['10_mv_create'])
execute_sql('ANALYZE mv_olist_customer_features')
print('MV populated; UNIQUE INDEX on customer_id created.')

```sql
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

CREATE MATERIALIZED VIEW mv_olist_customer_features AS
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id;

CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features (customer_id);
```

MV populated; UNIQUE INDEX on customer_id created.


In [17]:
SQL['12_mv_summary'] = """
SELECT COUNT(*) AS n_rows, COUNT(DISTINCT customer_id) AS n_keys, SUM(frequency) AS orders, SUM(monetary) AS monetary_brl, MIN(recency_days) AS min_recency, MAX(recency_days) AS max_recency, COUNT(*) FILTER (WHERE frequency <> 1) AS frequency_not_one, COUNT(*) FILTER (WHERE preferred_category='(none)') AS no_item_customers FROM mv_olist_customer_features;
""".strip()
run_sql('12_mv_summary', SQL['12_mv_summary'])

```sql
SELECT COUNT(*) AS n_rows, COUNT(DISTINCT customer_id) AS n_keys, SUM(frequency) AS orders, SUM(monetary) AS monetary_brl, MIN(recency_days) AS min_recency, MAX(recency_days) AS max_recency, COUNT(*) FILTER (WHERE frequency <> 1) AS frequency_not_one, COUNT(*) FILTER (WHERE preferred_category='(none)') AS no_item_customers FROM mv_olist_customer_features;
```

,value
metric,
n_rows,99441
n_keys,99441
orders,99441
monetary_brl,15843553.24
min_recency,1
max_recency,774
frequency_not_one,0
no_item_customers,775


[{'n_rows': 99441,
  'n_keys': 99441,
  'orders': Decimal('99441'),
  'monetary_brl': Decimal('15843553.24'),
  'min_recency': 1,
  'max_recency': 774,
  'frequency_not_one': 0,
  'no_item_customers': 775}]

In [18]:
SQL['16_mv_equivalence'] = "WITH recomputed AS (\nWITH\nreference_date AS (\n    -- Stage 0: stable reference date for historical dataset\n    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date\n    FROM olist_orders\n),\nstage1_orders_total AS (\n    -- Stage 1: orders + order_total + status\n    SELECT\n        o.customer_id,\n        o.order_id,\n        o.order_status,\n        o.order_purchase_timestamp AS order_ts,\n        o.order_purchase_timestamp::DATE AS order_date,\n        COALESCE(SUM(\n            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)\n        ), 0)::NUMERIC(12, 2) AS order_total\n    FROM olist_orders AS o\n    LEFT JOIN olist_order_items AS oi\n        ON oi.order_id = o.order_id\n    GROUP BY\n        o.customer_id,\n        o.order_id,\n        o.order_status,\n        o.order_purchase_timestamp\n),\nstage2_rfm AS (\n    -- Stage 2: RFM features per customer\n    SELECT\n        s.customer_id,\n        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,\n        COUNT(*) AS frequency,\n        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,\n        MIN(s.order_date) AS first_order_date\n    FROM stage1_orders_total AS s\n    CROSS JOIN reference_date AS rd\n    GROUP BY s.customer_id, rd.as_of_date\n),\nstage3_cohort AS (\n    -- Stage 3: cohort quarter from first order date\n    SELECT\n        r.*,\n        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter\n    FROM stage2_rfm AS r\n),\nstage4_rolling_aov AS (\n    -- Stage 4: rolling 30-day average order value per customer\n    SELECT\n        s.customer_id,\n        s.order_id,\n        s.order_ts,\n        s.order_total,\n        AVG(s.order_total) OVER (\n            PARTITION BY s.customer_id\n            ORDER BY s.order_ts\n            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW\n        )::NUMERIC(12, 2) AS avg_order_value_30d,\n        ROW_NUMBER() OVER (\n            PARTITION BY s.customer_id\n            ORDER BY s.order_ts DESC, s.order_id DESC\n        ) AS rn_recent\n    FROM stage1_orders_total AS s\n),\nstage5_cancel_rate AS (\n    -- Stage 5: cancellation rate per customer\n    SELECT\n        customer_id,\n        AVG(\n            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END\n        )::NUMERIC(6, 4) AS cancellation_rate\n    FROM stage1_orders_total\n    GROUP BY customer_id\n),\nstage6_category_revenue AS (\n    -- Stage 6: revenue by customer and product category\n    SELECT\n        o.customer_id,\n        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,\n        SUM(\n            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)\n        )::NUMERIC(12, 2) AS category_revenue\n    FROM olist_orders AS o\n    JOIN olist_order_items AS oi\n        ON oi.order_id = o.order_id\n    LEFT JOIN olist_products AS p\n        ON p.product_id = oi.product_id\n    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')\n),\nstage7_preferred_category AS (\n    -- Stage 7: preferred category per customer via top-1 ranking\n    SELECT\n        customer_id,\n        product_category_name,\n        category_revenue\n    FROM (\n        SELECT\n            cr.*,\n            ROW_NUMBER() OVER (\n                PARTITION BY cr.customer_id\n                ORDER BY cr.category_revenue DESC, cr.product_category_name\n            ) AS rn\n        FROM stage6_category_revenue AS cr\n    ) AS ranked\n    WHERE rn = 1\n)\nSELECT\n    c.customer_id,\n    c.customer_state,\n    COALESCE(coh.recency_days, 9999) AS recency_days,\n    COALESCE(coh.frequency, 0) AS frequency,\n    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,\n    coh.first_order_date,\n    coh.cohort_quarter,\n    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,\n    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,\n    COALESCE(pc.product_category_name, '(none)') AS preferred_category,\n    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue\nFROM olist_customers AS c\nLEFT JOIN stage3_cohort AS coh\n    ON coh.customer_id = c.customer_id\nLEFT JOIN stage4_rolling_aov AS aov\n    ON aov.customer_id = c.customer_id\n   AND aov.rn_recent = 1\nLEFT JOIN stage5_cancel_rate AS cancel\n    ON cancel.customer_id = c.customer_id\nLEFT JOIN stage7_preferred_category AS pc\n    ON pc.customer_id = c.customer_id\n), differences AS (\n(SELECT * FROM recomputed EXCEPT SELECT * FROM mv_olist_customer_features)\nUNION ALL\n(SELECT * FROM mv_olist_customer_features EXCEPT SELECT * FROM recomputed)\n)\nSELECT COUNT(*) AS difference_rows FROM differences;"
SQL['14_explain_lookup'] = 'EXPLAIN (ANALYZE, BUFFERS)\nWITH target_customer AS (\n    SELECT customer_id FROM olist_customers ORDER BY customer_id LIMIT 1\n)\nSELECT mv.* FROM mv_olist_customer_features AS mv\nJOIN target_customer AS t ON t.customer_id=mv.customer_id;'
SQL['15_explain_matched'] = "EXPLAIN (ANALYZE, BUFFERS)\nWITH\nreference_date AS (\n    -- Stage 0: stable reference date for historical dataset\n    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date\n    FROM olist_orders\n),\nstage1_orders_total AS (\n    -- Stage 1: orders + order_total + status\n    SELECT\n        o.customer_id,\n        o.order_id,\n        o.order_status,\n        o.order_purchase_timestamp AS order_ts,\n        o.order_purchase_timestamp::DATE AS order_date,\n        COALESCE(SUM(\n            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)\n        ), 0)::NUMERIC(12, 2) AS order_total\n    FROM olist_orders AS o\n    LEFT JOIN olist_order_items AS oi\n        ON oi.order_id = o.order_id\n    GROUP BY\n        o.customer_id,\n        o.order_id,\n        o.order_status,\n        o.order_purchase_timestamp\n),\nstage2_rfm AS (\n    -- Stage 2: RFM features per customer\n    SELECT\n        s.customer_id,\n        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,\n        COUNT(*) AS frequency,\n        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,\n        MIN(s.order_date) AS first_order_date\n    FROM stage1_orders_total AS s\n    CROSS JOIN reference_date AS rd\n    GROUP BY s.customer_id, rd.as_of_date\n),\nstage3_cohort AS (\n    -- Stage 3: cohort quarter from first order date\n    SELECT\n        r.*,\n        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter\n    FROM stage2_rfm AS r\n),\nstage4_rolling_aov AS (\n    -- Stage 4: rolling 30-day average order value per customer\n    SELECT\n        s.customer_id,\n        s.order_id,\n        s.order_ts,\n        s.order_total,\n        AVG(s.order_total) OVER (\n            PARTITION BY s.customer_id\n            ORDER BY s.order_ts\n            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW\n        )::NUMERIC(12, 2) AS avg_order_value_30d,\n        ROW_NUMBER() OVER (\n            PARTITION BY s.customer_id\n            ORDER BY s.order_ts DESC, s.order_id DESC\n        ) AS rn_recent\n    FROM stage1_orders_total AS s\n),\nstage5_cancel_rate AS (\n    -- Stage 5: cancellation rate per customer\n    SELECT\n        customer_id,\n        AVG(\n            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END\n        )::NUMERIC(6, 4) AS cancellation_rate\n    FROM stage1_orders_total\n    GROUP BY customer_id\n),\nstage6_category_revenue AS (\n    -- Stage 6: revenue by customer and product category\n    SELECT\n        o.customer_id,\n        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,\n        SUM(\n            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)\n        )::NUMERIC(12, 2) AS category_revenue\n    FROM olist_orders AS o\n    JOIN olist_order_items AS oi\n        ON oi.order_id = o.order_id\n    LEFT JOIN olist_products AS p\n        ON p.product_id = oi.product_id\n    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')\n),\nstage7_preferred_category AS (\n    -- Stage 7: preferred category per customer via top-1 ranking\n    SELECT\n        customer_id,\n        product_category_name,\n        category_revenue\n    FROM (\n        SELECT\n            cr.*,\n            ROW_NUMBER() OVER (\n                PARTITION BY cr.customer_id\n                ORDER BY cr.category_revenue DESC, cr.product_category_name\n            ) AS rn\n        FROM stage6_category_revenue AS cr\n    ) AS ranked\n    WHERE rn = 1\n)\nSELECT\n    c.customer_id,\n    c.customer_state,\n    COALESCE(coh.recency_days, 9999) AS recency_days,\n    COALESCE(coh.frequency, 0) AS frequency,\n    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,\n    coh.first_order_date,\n    coh.cohort_quarter,\n    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,\n    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,\n    COALESCE(pc.product_category_name, '(none)') AS preferred_category,\n    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue\nFROM olist_customers AS c\nLEFT JOIN stage3_cohort AS coh\n    ON coh.customer_id = c.customer_id\nLEFT JOIN stage4_rolling_aov AS aov\n    ON aov.customer_id = c.customer_id\n   AND aov.rn_recent = 1\nLEFT JOIN stage5_cancel_rate AS cancel\n    ON cancel.customer_id = c.customer_id\nLEFT JOIN stage7_preferred_category AS pc\n    ON pc.customer_id = c.customer_id\nWHERE c.customer_id=(\n    SELECT customer_id FROM olist_customers ORDER BY customer_id LIMIT 1\n);"
validate_features(); validate_stages_and_lookup()

,value
metric,
n_rows,99441
n_keys,99441
orders,99441
monetary_brl,15843553.24
min_recency,1
max_recency,774
frequency_not_one,0
no_item_customers,775


### CONCURRENT REFRESH — окрема SQL-клітинка після UNIQUE INDEX

In [19]:
SQL['11_mv_refresh'] = """
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;
""".strip()
sql_display(SQL['11_mv_refresh']); execute_sql(SQL['11_mv_refresh']); print('11_mv_refresh: виконано.')

```sql
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;
```

11_mv_refresh: виконано.


In [20]:
require('Full pipeline equals MV after concurrent refresh',scalar(SQL['16_mv_equivalence'])==0)
print('Full MV == full recomputation after concurrent refresh.')

Full MV == full recomputation after concurrent refresh.


mv_olist_customer_features матеріалізує **весь** CTE pipeline: 99 441 customer_id та 11 колонок. Повний UNIQUE INDEX на customer_id підтримує CONCURRENT refresh і lookup. Сама зміна базової таблиці не оновлює MV: refresh fixture показує застарілу суму, нове значення після REFRESH MATERIALIZED VIEW CONCURRENTLY та rollback до оригіналу. Рівність pipeline і MV перевірено через EXCEPT в обох напрямках; fixtures не залишають тестових змін. Це кешування всього набору ознак, а не лише найдорожчої колонки.

## 6. EXPLAIN (ANALYZE, BUFFERS)
Для required comparison збережено повне переобчислення всіх features на всіх customer_id та indexed MV lookup одного customer_id. Це різні обсяги результату. Третій план порівнює той самий customer_id й одинадцять однакових колонок. Усі SQL виконано реально; cache попередньо прогрітий завантаженням і перевірками.

Для кожного сценарію text та JSON плани — **два окремі виконання**, тому їхні execution times можуть відрізнятися. Таблиця й Reflection використовують primary text-plan timings; JSON дає inspectable структуру другого запуску. Один локальний вимір не є універсальним benchmark.

In [21]:
SQL['13_explain_pipeline'] = """
EXPLAIN (ANALYZE, BUFFERS)
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id;
""".strip()
require('Full EXPLAIN query equals canonical pipeline',SQL['13_explain_pipeline'].split('\n',1)[1].strip()==PIPELINE_SQL.strip())
explain_profile('full_pipeline',SQL['13_explain_pipeline'])

```sql
EXPLAIN (ANALYZE, BUFFERS)
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id;
```

{'scenario': 'full_pipeline',
 'text_execution_ms': 597.659,
 'text_planning_ms': 0.39,
 'json_execution_ms': 611.2,
 'actual_rows_json': 99441}

In [22]:
explain_profile('mv_lookup',SQL['14_explain_lookup'])

```sql
EXPLAIN (ANALYZE, BUFFERS)
WITH target_customer AS (
    SELECT customer_id FROM olist_customers ORDER BY customer_id LIMIT 1
)
SELECT mv.* FROM mv_olist_customer_features AS mv
JOIN target_customer AS t ON t.customer_id=mv.customer_id;
```

{'scenario': 'mv_lookup',
 'text_execution_ms': 0.015,
 'text_planning_ms': 0.098,
 'json_execution_ms': 0.008,
 'actual_rows_json': 1}

In [23]:
explain_profile('matched_pipeline_lookup',SQL['15_explain_matched'])
require('Full plan actually emits every customer',plans['full_pipeline']['json']['Plan']['Actual Rows']==99441)
require('Lookup plans each emit one identical customer',plans['mv_lookup']['json']['Plan']['Actual Rows']==plans['matched_pipeline_lookup']['json']['Plan']['Actual Rows']==1)
require('MV lookup uses unique customer index',any(n.get('Index Name')=='idx_mv_olist_customer_features_customer_id' and n['Node Type']=='Index Scan' for p,n in walk_plan(plans['mv_lookup']['json']['Plan'])))
table_display(profile_rows)

```sql
EXPLAIN (ANALYZE, BUFFERS)
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id
WHERE c.customer_id=(
    SELECT customer_id FROM olist_customers ORDER BY customer_id LIMIT 1
);
```

scenario,text_execution_ms,text_planning_ms,json_execution_ms,actual_rows_json
full_pipeline,597.659,0.390,611.200,99441
mv_lookup,0.015,0.098,0.008,1
matched_pipeline_lookup,103.464,0.389,102.755,1


In [24]:
text_full=plans['full_pipeline']['text']
window_pattern=r'WindowAgg.*?actual time=[\d.]+\.\.([\d.]+)'
rolling_text_ms=float(re.search(window_pattern,text_full[text_full.index('Subquery Scan on aov'):]).group(1))
category_text_ms=float(re.search(window_pattern,text_full[text_full.index('Subquery Scan on ranked'):]).group(1))
disk_sizes=[int(v) for v in re.findall(r'Disk: (\d+)kB',text_full)]
quality=results['quality']['rows'][0] if 'quality' in results else results['04_quality']['rows'][0]
context={'full_ms':plans['full_pipeline']['text_execution_ms'],'lookup_ms':plans['mv_lookup']['text_execution_ms'],'matched_ms':plans['matched_pipeline_lookup']['text_execution_ms'],
         'rolling_text_ms':rolling_text_ms,'category_text_ms':category_text_ms,'sort_disk_kb':max(disk_sizes),
         'expensive_stage':'Stage 4: rolling AOV та вибір останнього замовлення','n_customers':quality['customers'],'n_buyers':quality['real_buyers'],'n_items':quality['order_items'],
         'total_brl':'15 843 553,24','reference_date':str(quality['reference_date']),'n_checks':len(checks),'postgres_version':'16.2'}
print('Primary text timings (ms):', {k:context[k] for k in ['full_ms','matched_ms','lookup_ms']})
print('WindowAgg subtree inclusive timings (ms):',context['rolling_text_ms'],context['category_text_ms'])

Primary text timings (ms): {'full_ms': 597.659, 'matched_ms': 103.464, 'lookup_ms': 0.015}
WindowAgg subtree inclusive timings (ms): 210.948 140.919


In [25]:
display(Markdown('Фактичний текстовий EXPLAIN (ANALYZE, BUFFERS): повний pipeline — **{full_ms:.3f} мс**, один MV lookup — **{lookup_ms:.3f} мс**, переобчислення pipeline для того самого customer_id — **{matched_ms:.3f} мс**. Перші два сценарії мають різний output volume: 99 441 рядок проти одного. Додатковий третій сценарій повертає той самий customer та всі 11 колонок, що й MV lookup.\n\nУ повному плані видно Seq Scan, JOIN, Aggregate, Sort та WindowAgg. Найбільше помітне піддерево ознак — {expensive_stage}; час включає дочірні операції й не є незалежною сумою витрат колонки. Сортування external merge має запис на диск; MV lookup використовує Index Scan по customer_id. Окремі текстовий і JSON EXPLAIN є двома виконаннями, тому їхні часи можуть відрізнятися; висновки про runtime в цьому тексті беруться з текстового плану. Результат одного локального запуску залежить від кешу, обладнання, planner settings та паралелізму; числовий speedup не переноситься автоматично на інші системи.'.format_map(context)))

Фактичний текстовий EXPLAIN (ANALYZE, BUFFERS): повний pipeline — **597.659 мс**, один MV lookup — **0.015 мс**, переобчислення pipeline для того самого customer_id — **103.464 мс**. Перші два сценарії мають різний output volume: 99 441 рядок проти одного. Додатковий третій сценарій повертає той самий customer та всі 11 колонок, що й MV lookup.

У повному плані видно Seq Scan, JOIN, Aggregate, Sort та WindowAgg. Найбільше помітне піддерево ознак — Stage 4: rolling AOV та вибір останнього замовлення; час включає дочірні операції й не є незалежною сумою витрат колонки. Сортування external merge має запис на диск; MV lookup використовує Index Scan по customer_id. Окремі текстовий і JSON EXPLAIN є двома виконаннями, тому їхні часи можуть відрізнятися; висновки про runtime в цьому тексті беруться з текстового плану. Результат одного локального запуску залежить від кешу, обладнання, planner settings та паралелізму; числовий speedup не переноситься автоматично на інші системи.

## 7. Інваріанти та контрольні fixtures
Оригінальний Olist має frequency=1 на customer_id, тому він не перевіряє складне rolling-вікно. Контрольні записи з відомими ручними значеннями окремо перевіряють межі 30 днів, timestamp peers, deterministic latest row, all-status cancellation denominator, COALESCE та category ties. Кожний fixture відкотиться. Негативні tests очікують конкретні SQLSTATE. Cycle fixtures завжди захищено visited path, depth guard та statement timeout; незахищену нескінченну рекурсію не запускаємо.

In [26]:
validate_pipeline_boundaries()

customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
hw5_test_a,SP,259,4,100.00,2018-01-01,2018-01-01,30.00,0.2500,alpha,100.00
hw5_test_none,RJ,9999,0,0.00,None,None,0.00,0.0000,(none),0.00
hw5_test_null,MG,289,1,12.00,2018-01-02,2018-01-01,12.00,0.0000,(unknown),12.00
hw5_test_tie,BA,288,1,20.00,2018-01-03,2018-01-01,20.00,0.0000,alpha,10.00


In [27]:
validate_recursion()

Цикл A ↔ B: root-only traversal повернув R, C, I; A/B пропущені. All-node seed повернув 8 path-рядків, depth ≤ 2. Ланцюг із 55 вузлів обрізано до depth=50.


In [28]:
validate_constraints()

fixture,expected_sqlstate,actual_sqlstate
Duplicate customer key,23505,23505
Invalid order status,23514,23514
Negative item price,23514,23514
Missing customer foreign key,23503,23503
Self-referral,23514,23514
MV missing UNIQUE index,55000,55000


In [29]:
validate_refresh_change()

customer_id,before_brl,stale_brl,changed_source_brl,refreshed_brl
3ce436f183e68e07877b285a838db11a,72.19,72.19,73.19,73.19


Транзакцію відкочено; вихідні дані та початковий MV відновлено.


In [30]:
context['n_checks']=len(checks)
require('All notebook checks passed',all(c['passed'] for c in checks))
context['n_checks']=len(checks)
print(f"PASS: {len(checks)} assertions; all full-source and boundary checks passed.")
version_rows=rows("SELECT name,setting,unit FROM pg_settings WHERE name IN ('server_version','work_mem','shared_buffers','max_parallel_workers_per_gather','jit','enable_hashagg','random_page_cost') ORDER BY name")
table_display(version_rows)

PASS: 74 assertions; all full-source and boundary checks passed.


name,setting,unit
enable_hashagg,on,None
jit,on,None
max_parallel_workers_per_gather,2,None
random_page_cost,4,None
server_version,16.2,None
shared_buffers,16384,8kB
work_mem,4096,kB


Перевірки notebook охоплюють вихідні bytes/headers/counts, raw/typed відповідність, ключі й constraints, all-status monetary із freight, покриття customer_id, рівність повного pipeline та MV, індекс і refresh, межі 30-денного RANGE та timestamp peers, latest-order і category ties, клієнта без замовлень, NULL-компоненти, цикл і depth guard. Fixtures відкочуються; PostgreSQL працює у власному тимчасовому локальному кластері.

## 8. Reflection

In [31]:
reflection_template = '**Reflection**\n\n1. За EXPLAIN найдорожче помітне піддерево — {expensive_stage}: WindowAgg обчислює AVG і ROW_NUMBER, Sort використовує external merge. RFM і cancellation мають HashAggregate; джерела — Seq Scan, MV lookup — Index Scan. Часи піддерев включають дочірню роботу; їх не можна додавати або називати CPU-часом ознаки. Повний pipeline повернув 99 441 рядок за {full_ms:.3f} мс; MV lookup — один за {lookup_ms:.3f} мс. Переобчислення того самого customer_id зайняло {matched_ms:.3f} мс. Це один запуск, залежний від кешу й середовища.\n\n2. Матеріалізація доречна для повторного serving: вона переносить JOIN, агрегації та сортування на refresh, а UNIQUE INDEX прискорює lookup. Обираю щодобове оновлення для пакетної сегментації клієнтів: добове запізнення прийнятне, а рідший цикл скорочує кількість повних перерахунків. Потрібно контролювати завершення refresh і вік кешу. Для незмінного історичного архіву повторне оновлення не додає даних.\n\n3. Звичайний VIEW доречний для нечастих аналітичних запитів або коли потрібні актуальні committed-дані на момент snapshot запиту. Він повторює обчислення при читанні й не зберігає готовий результат; тому часті дорогі lookup збільшують навантаження.\n\n4. Recursive CTE починає з коренів через NOT EXISTS. Масив path_ids блокує повторне відвідування вузла, а depth < 50 обмежує довжину обходу. При досяжному циклі A → B → A повторний A не додається. Однак відокремлений цикл без кореня взагалі не потрапляє до anchor; завершення обходу не доводить відсутності циклів. Для перевірки всього графа потрібні інші стартові вузли або окрема глобальна валідація. Depth guard також може відсікти коректний довгий ланцюг.\n\n5. У цьому pipeline пріоритетним для мене є recompute-cost: важливо уникати зайвих повних перерахунків. MV дозволяє повторно використовувати ознаки між щодобовими refresh ціною storage та добової затримки актуальності. Цей pipeline описує стан наприкінці датасету; для навчання без витоку потрібні ознаки на дату прогнозу та доступність подій на той момент.\n'
reflection_rendered=reflection_template.format_map(context)
display(Markdown(reflection_rendered))
(WORK_DIR/'reflection-rendered.md').write_text(reflection_rendered,encoding='utf-8')

**Reflection**

1. За EXPLAIN найдорожче помітне піддерево — Stage 4: rolling AOV та вибір останнього замовлення: WindowAgg обчислює AVG і ROW_NUMBER, Sort використовує external merge. RFM і cancellation мають HashAggregate; джерела — Seq Scan, MV lookup — Index Scan. Часи піддерев включають дочірню роботу; їх не можна додавати або називати CPU-часом ознаки. Повний pipeline повернув 99 441 рядок за 597.659 мс; MV lookup — один за 0.015 мс. Переобчислення того самого customer_id зайняло 103.464 мс. Це один запуск, залежний від кешу й середовища.

2. Матеріалізація доречна для повторного serving: вона переносить JOIN, агрегації та сортування на refresh, а UNIQUE INDEX прискорює lookup. Обираю щодобове оновлення для пакетної сегментації клієнтів: добове запізнення прийнятне, а рідший цикл скорочує кількість повних перерахунків. Потрібно контролювати завершення refresh і вік кешу. Для незмінного історичного архіву повторне оновлення не додає даних.

3. Звичайний VIEW доречний для нечастих аналітичних запитів або коли потрібні актуальні committed-дані на момент snapshot запиту. Він повторює обчислення при читанні й не зберігає готовий результат; тому часті дорогі lookup збільшують навантаження.

4. Recursive CTE починає з коренів через NOT EXISTS. Масив path_ids блокує повторне відвідування вузла, а depth < 50 обмежує довжину обходу. При досяжному циклі A → B → A повторний A не додається. Однак відокремлений цикл без кореня взагалі не потрапляє до anchor; завершення обходу не доводить відсутності циклів. Для перевірки всього графа потрібні інші стартові вузли або окрема глобальна валідація. Depth guard також може відсікти коректний довгий ланцюг.

5. У цьому pipeline пріоритетним для мене є recompute-cost: важливо уникати зайвих повних перерахунків. MV дозволяє повторно використовувати ознаки між щодобовими refresh ціною storage та добової затримки актуальності. Цей pipeline описує стан наприкінці датасету; для навчання без витоку потрібні ознаки на дату прогнозу та доступність подій на той момент.


## 9. Збереження результатів і cleanup
`work/query-results.json` містить результати, перевірки, параметри та timings; `work/plans/` — фактичні text і JSON plans.

In [32]:
write_json(WORK_DIR/'query-results.json',{'results': results, 'checks': checks, 'profiles': profile_rows, 'context': context, 'postgres_version': postgres_version, 'package_versions': package_versions, 'settings': version_rows, 'source_hashes': SOURCE_HASHES})
cleanup_hw5()
print('Результати збережено. Власний PostgreSQL зупинено; власний temporary directory видалено.')

Результати збережено. Власний PostgreSQL зупинено; власний temporary directory видалено.
